In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'

metadata_df

,sample_title,subject status,tissue,tissue type
samples,,,,
GSM4521251,ST1,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521252,ST2,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521253,ST3,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521254,ST4,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521255,ST5,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521256,ST6,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521257,ST7,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521258,ST8,short-term (ST) PDAC survivor,Pancreas,Tumor
GSM4521259,ST9,short-term (ST) PDAC survivor,Pancreas,Tumor


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()
# df_combined.to_csv(f'{GEO_ID}_raw_counts_NCBI.csv', index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_95835/4076848676.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,subject status,tissue,tissue type,DDX11L1,WASH7P,MIR6859-1,MIR1302-2HG,MIR1302-2,FAM138A,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM4521251,ST1,short-term (ST) PDAC survivor,Pancreas,Tumor,7,717,6,0,3,9,...,2738,0,1,0,1914,576,14,766,0,9
GSM4521252,ST2,short-term (ST) PDAC survivor,Pancreas,Tumor,13,1879,7,0,0,1,...,375,3,1,1,324,68,9,99,0,1
GSM4521253,ST3,short-term (ST) PDAC survivor,Pancreas,Tumor,30,3133,14,0,0,26,...,315,1,1,0,327,80,3,142,0,3
GSM4521254,ST4,short-term (ST) PDAC survivor,Pancreas,Tumor,16,944,5,0,2,0,...,894,0,0,0,784,213,17,375,3,9
GSM4521255,ST5,short-term (ST) PDAC survivor,Pancreas,Tumor,13,841,4,0,9,2,...,1153,3,0,0,950,295,24,371,0,8
GSM4521256,ST6,short-term (ST) PDAC survivor,Pancreas,Tumor,4,1323,9,0,0,0,...,1593,1,0,0,1072,279,13,668,1,9
GSM4521257,ST7,short-term (ST) PDAC survivor,Pancreas,Tumor,30,2635,14,0,1,17,...,1699,3,4,2,1559,440,39,599,2,8
GSM4521258,ST8,short-term (ST) PDAC survivor,Pancreas,Tumor,5,343,1,0,4,6,...,788,1,0,1,788,307,32,272,1,8
GSM4521259,ST9,short-term (ST) PDAC survivor,Pancreas,Tumor,2,884,4,0,0,0,...,1414,1,1,1,1113,286,31,400,1,6


# Parse sumplementary files to get clinical data


Download this PDF and save in this folder: https://pmc.ncbi.nlm.nih.gov/articles/instance/9247075/bin/41598_2022_14592_MOESM1_ESM.pdf

In [3]:
import pymupdf

from openai import OpenAI
from typing import Optional, List, Dict, Any

client = OpenAI(api_key='os.environ.get("OPENAI_API_KEY")')


def call_openai(
    user_text: str,
    *,
    system_instructions: Optional[str] = None,
    model: str = "gpt-5-mini-2025-08-07",
    verbosity: str = "low",
    reasoning_effort: str = "low",
) -> str:
    
    messages: List[Dict[str, Any]] = []
    if system_instructions:
        messages.append({
            "role": "system",
            "content": [{"type": "input_text", "text": system_instructions}],
        })
    messages.append({
        "role": "user",
        "content": [{"type": "input_text", "text": user_text}],
    })

    resp = client.responses.parse(
        model=model,
        input=messages,
        text={"verbosity": verbosity},
        reasoning={"effort": reasoning_effort},
        service_tier="flex",
    )
    return resp


prompt = """Extract TABLE S1 from the following PDF text. The table may be split across multiple pages/parts. 
Combine all parts into a single unified CSV format.

Instructions:
- Identify all parts of TABLE S1 in the text
- Extract the table headers and all data rows
- Merge the parts into one complete table
- Output as valid CSV format with proper column headers
- Preserve all data values exactly as they appear

PDF Text:
{text}

Output the complete TABLE S1 as CSV:"""


# doc = pymupdf.open("41598_2022_14592_MOESM1_ESM.pdf")
# # Extract text from first 4 pages
# pdf_text = "\n\n".join([page.get_text() for page in doc[1:4]])
# doc.close()


# result = call_openai(
#     prompt.format(text=pdf_text),
#     system_instructions="Follow the instructions carefully. Do only what is asked.",
#     reasoning_effort="medium"
# )

# # Extract text from response
# if result.output and len(result.output) > 1:
#     message = result.output[1]
#     if hasattr(message, 'content') and message.content:   
#         # Save to CSV
#         with open('table_s1_clinical_data.csv', 'w') as f:
#             f.write(message.content[0].text)

In [4]:
df_clinical_data = pd.read_csv('table_s1_clinical_data.csv')

# Create merge key: insert "T" after first letter (S1 -> ST1, L1 -> LT1)
df_clinical_data['sample_title'] = df_clinical_data['Patient ID'].str[0] + 'T' + df_clinical_data['Patient ID'].str[1:]


# Merge the dataframes
df_combined = df_clinical_data.merge(
    df_combined,
    left_on='sample_title',
    right_on='sample_title',
    how='inner'
)


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.to_csv(data_file_name, index=False)
df_combined


,Patient ID,Group*,Age at the surgery (years),Sexa,Tumor size (cm)b (measured after surgery),T,N,Number of nodes analyzed,ratio node (invaded/analyzed),ratio node % (invaded/analysed),...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
0,S1,ST,74,F,3.0,4,1,25,2/25,0.08,...,2738,0,1,0,1914,576,14,766,0,9
1,S2,ST,71,M,3.5,3,0,12,0/12,0.00,...,375,3,1,1,324,68,9,99,0,1
2,S3,ST,72,F,3.2,3,0,7,0/7,0.00,...,315,1,1,0,327,80,3,142,0,3
3,S4,ST,73,M,4.0,3,1,7,5/7,0.71,...,894,0,0,0,784,213,17,375,3,9
4,S5,ST,54,F,3.2,4,1,12,7/12,0.58,...,1153,3,0,0,950,295,24,371,0,8
5,S6,ST,80,M,4.0,3,1,8,2/8,0.25,...,1593,1,0,0,1072,279,13,668,1,9
6,S7,ST,80,M,4.0,3,0,16,0/16,0.00,...,1699,3,4,2,1559,440,39,599,2,8
7,S8,ST,63,F,3.0,4,1,14,4/14,0.28,...,788,1,0,1,788,307,32,272,1,8
8,S9,ST,66,M,5.0,3,1B,16,10/16,0.62,...,1414,1,1,1,1113,286,31,400,1,6
9,S10,ST,66,M,5.0,4,1,4,0/4,0.00,...,2293,0,0,0,1383,350,21,690,0,4


# Generate description


In [5]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_supplement

description_file_path = 'description.json'

if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_supplement(GEO_ID, 'table_s1_clinical_data.csv')
    print("Description generated")

desc_data['data_summary'] = {
    'samples': df_clinical_data.shape[0],
    'clinical_features': df_combined.shape[1] - df_raw_counts.shape[1],
    'genes': df_raw_counts.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)


Description updated
